In [3]:
import pandas as pd

In [5]:
# Load  original Skytrax scraped data
skytrax_df = pd.read_csv("../data/cleaned_reviews.csv")

# Load the  Kaggle dataset
kaggle_df = pd.read_csv("../data/kaggle_reviews.csv")



In [6]:
# Check both loaded correctly
print("Skytrax shape:", skytrax_df.shape)
print("Kaggle shape:", kaggle_df.shape)

# Preview both dataset
print("\nSkytrax columns:", skytrax_df.columns.tolist())
print("Kaggle columns:", kaggle_df.columns.tolist())

Skytrax shape: (3995, 7)
Kaggle shape: (3701, 20)

Skytrax columns: ['review_text', 'rating', 'date', 'verified', 'country', 'clean_text', 'sentiment']
Kaggle columns: ['Unnamed: 0', 'OverallRating', 'ReviewHeader', 'Name', 'Datetime', 'VerifiedReview', 'ReviewBody', 'TypeOfTraveller', 'SeatType', 'Route', 'DateFlown', 'SeatComfort', 'CabinStaffService', 'GroundService', 'ValueForMoney', 'Recommended', 'Aircraft', 'Food&Beverages', 'InflightEntertainment', 'Wifi&Connectivity']


## Standadise the Kaggle Dataset

In [7]:
# Rename Kaggle columns to match  Skytrax column names
kaggle_clean = kaggle_df.rename(columns={
    'ReviewBody':      'review_text',
    'OverallRating':   'rating',
    'Datetime':        'date',
    'VerifiedReview':  'verified'
})

# Keep only the columns needed and drop the rest
kaggle_clean = kaggle_clean[[
    'review_text',
    'rating',
    'date',
    'verified'
]]

# Add a source column so we know where each review came from
kaggle_clean['source'] = 'kaggle'

print("Kaggle cleaned shape:", kaggle_clean.shape)
print("\nFirst few rows:")
kaggle_clean.head()

Kaggle cleaned shape: (3701, 5)

First few rows:


,review_text,rating,date,verified,source
0,4 Hours before takeoff we received a Mail stat...,1.0,19th November 2023,True,kaggle
1,I recently had a delay on British Airways from...,3.0,19th November 2023,True,kaggle
2,"Boarded on time, but it took ages to get to th...",8.0,16th November 2023,False,kaggle
3,"5 days before the flight, we were advised by B...",1.0,16th November 2023,True,kaggle
4,"We traveled to Lisbon for our dream vacation, ...",1.0,14th November 2023,False,kaggle


## Standardize the Skytrax Dataset

In [8]:
# Add a source column to Skytrax data also
skytrax_df['source'] = 'skytrax'

# Keep only the matching columns
skytrax_clean = skytrax_df[[
    'review_text',
    'rating',
    'date',
    'verified',
    'source'
]]

print("Skytrax cleaned shape:", skytrax_clean.shape)
print("\nFirst few rows:")
skytrax_clean.head()

Skytrax cleaned shape: (3995, 5)

First few rows:


,review_text,rating,date,verified,source
0,✅Trip Verified| I'm not entirely sure as to wh...,8,2026-05-02,True,skytrax
1,✅Trip Verified| Our first time in the new bu...,8,2026-04-21,True,skytrax
2,✅Trip Verified| Highly commendable again on ...,10,2026-04-09,True,skytrax
3,✅Trip Verified| Highly commendable on all fr...,10,2026-04-09,True,skytrax
4,Not Verified| Although the staff on board t...,2,2026-04-06,False,skytrax


## Combine both dataset

In [9]:
# Stack both datasets on top of each other
combined_df = pd.concat(
    [skytrax_clean, kaggle_clean],
    ignore_index=True
)

print(f"Combined dataset shape: {combined_df.shape}")
print(f"\nSource breakdown:")
print(combined_df['source'].value_counts())

Combined dataset shape: (7696, 5)

Source breakdown:
source
skytrax    3995
kaggle     3701
Name: count, dtype: int64


## Clean the combine dataset

In [10]:
import re

# Drop rows where review text or rating is missing
combined_df = combined_df.dropna(subset=['review_text', 'rating'])

# Convert rating to integer
combined_df['rating'] = combined_df['rating'].astype(int)

# Remove duplicate reviews — same review appearing in both datasets
combined_df = combined_df.drop_duplicates(subset=['review_text'])

# Clean the review text 
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'✓ Trip Verified \|', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

combined_df['clean_text'] = combined_df['review_text'].apply(clean_text)

# Remove very short reviews
combined_df = combined_df[combined_df['clean_text'].str.split().str.len() >= 10]

print(f"Final combined dataset shape: {combined_df.shape}")
print(f"\nRating distribution:")
print(combined_df['rating'].value_counts().sort_index())

Final combined dataset shape: (6287, 6)

Rating distribution:
rating
1     1657
2      752
3      713
4      412
5      383
6      305
7      482
8      590
9      477
10     516
Name: count, dtype: int64


## Assign Sentiment Labels

In [11]:
# label the ratings
def assign_sentiment(rating):
    if rating <= 3:
        return 'negative'
    elif rating <= 6:
        return 'neutral'
    else:
        return 'positive'

combined_df['sentiment'] = combined_df['rating'].apply(assign_sentiment)

print("Sentiment distribution:")
print(combined_df['sentiment'].value_counts())

# Quick percentage breakdown
total = len(combined_df)
for sentiment in ['negative', 'neutral', 'positive']:
    count = (combined_df['sentiment'] == sentiment).sum()
    print(f"{sentiment}: {count} ({count/total*100:.1f}%)")

Sentiment distribution:
sentiment
negative    3122
positive    2065
neutral     1100
Name: count, dtype: int64
negative: 3122 (49.7%)
neutral: 1100 (17.5%)
positive: 2065 (32.8%)


In [12]:
# Save to a new file — never overwrite your original cleaned data
combined_df.to_csv('../data/combined_reviews.csv', index=False)

print(f"Saved {len(combined_df)} combined reviews to data/combined_reviews.csv")

Saved 6287 combined reviews to data/combined_reviews.csv
